# Script 3 — Person-level merge of the raw NHANES files

Builds the person-level master file by loading eight raw NHANES files for
each of the two cycles (2011–2012, suffix `G`; 2013–2014, suffix `H`),
left-joining them on `SEQN` within a cycle, and stacking the two cycles.
Columns that a right-hand file shares with the left-hand frame are dropped
before the join, so no suffixed duplicates arise.

## Inputs

In `data/raw/2011-2012/` and `data/raw/2013-2014/` (file names matched
without regard to case):

- `DEMO` — demographics
- `CFQ` — cognitive functioning
- `SLQ` — sleep
- `DPQ` — depression screener (PHQ-9)
- `PAQ` — physical activity questionnaire
- `PAXDAY` — accelerometer day-level summaries (one row per participant-day)
- `PAXHD` — accelerometer header
- `BMX` — body measures

## Output

- `data/processed/0_check_personlevel.csv` — one row per participant-day of
  `PAXDAY` for participants with accelerometry, otherwise one row per
  participant; script 4 reduces it to one row per participant.

## Step 1 — Imports


In [ ]:
import pandas as pd
from pyreadstat import read_xport
from config import raw_2011, raw_2013, processed, raw_file

## Step 2 — Helper functions

`load_xpt` reads one SAS transport file. `merge_keep_left` left-joins two
frames on `SEQN`, dropping columns of the right frame that already exist in
the left frame.

In [ ]:
def load_xpt(folder, filename):
    path = raw_file(folder, filename)
    df, meta = read_xport(str(path))
    print(f'loaded {path.name}  shape = {df.shape}')
    return df


def merge_keep_left(base, add):
    if 'SEQN' not in add.columns:
        raise ValueError('SEQN not found in file being merged')
    drop_cols = [c for c in add.columns if c in base.columns and c != 'SEQN']
    add = add.drop(columns=drop_cols)
    return base.merge(add, on='SEQN', how='left')

## Step 3 — Build one cycle's person-level frame

For a cycle suffix (`G` or `H`), loads the eight raw files, merges them into one
frame and adds a `CYCLE` column.


In [ ]:
def build_cycle(folder, suffix):
    demo   = load_xpt(folder, f'DEMO_{suffix}.XPT')
    cfq    = load_xpt(folder, f'CFQ_{suffix}.XPT')
    slq    = load_xpt(folder, f'SLQ_{suffix}.XPT')
    dpq    = load_xpt(folder, f'DPQ_{suffix}.XPT')
    paq    = load_xpt(folder, f'PAQ_{suffix}.XPT')
    paxday = load_xpt(folder, f'PAXDAY_{suffix}.XPT')
    paxhd  = load_xpt(folder, f'PAXHD_{suffix}.XPT')
    bmi    = load_xpt(folder, f'BMX_{suffix}.XPT')

    merged = demo
    for add in [cfq, slq, dpq, paq, paxday, paxhd, bmi]:
        merged = merge_keep_left(merged, add)
    merged['CYCLE'] = '2011-2012' if suffix == 'G' else '2013-2014'
    return merged


## Step 4 — Build both cycles and write the master file

Builds the 2011–2012 and 2013–2014 frames, stacks them and writes
`data/processed/0_check_personlevel.csv`.

In [ ]:
df_g = build_cycle(raw_2011, 'G')
df_h = build_cycle(raw_2013, 'H')

master = pd.concat([df_g, df_h], ignore_index=True)
processed.mkdir(parents=True, exist_ok=True)
out = processed / '0_check_personlevel.csv'
master.to_csv(out, index=False)

print(f'wrote {out.name}: {len(master):,} rows x {master.shape[1]} columns, '
      f'{master["SEQN"].nunique():,} participants')